# Web search tool with the Perplexity Search API

The [Perplexity Search API](https://docs.perplexity.ai/docs/search/quickstart) returns ranked web results with titles, URLs, publication dates, and extracted page text. This notebook shows how to use it as a function tool for an OpenAI model through the Responses API, so the model can look up current information and cite the pages it used.

You will:

1. Call the Search API directly with the Perplexity Python library.
2. Wrap the call as a function tool and run a short tool-calling loop with the Responses API.
3. Restrict results by domain and recency.

## Prerequisites

- An OpenAI API key in the `OPENAI_API_KEY` environment variable.
- A Perplexity API key in the `PERPLEXITY_API_KEY` environment variable. Create one in the [Perplexity API portal](https://console.perplexity.ai/project/keys).
- Python 3.10 or later.
- `RUN_LIVE=1` in the environment. The cells that call either service are gated behind this flag, so the notebook can be opened and its structure checked without making paid requests.

Calls to both services incur usage charges. See [OpenAI pricing](https://openai.com/api/pricing/) and [Perplexity API pricing](https://docs.perplexity.ai/docs/getting-started/pricing) before setting `RUN_LIVE=1`.

## 1. Install dependencies

The notebook needs the OpenAI Python SDK and the Perplexity Python library (`perplexityai`).

In [ ]:
%pip install --quiet --upgrade openai perplexityai

## 2. Create the clients

Both SDKs read their API keys from the environment. Do not paste keys into the notebook. The clients are created only when `RUN_LIVE=1`, and the later cells skip their live requests when the flag is not set.

The `default_headers` argument adds two optional headers to every Search API request: `X-Pplx-Integration`, which identifies the integration that sent the request, and a `User-Agent` that names this cookbook while keeping the SDK version.

In [ ]:
import os

import perplexity
from openai import OpenAI
from perplexity import Perplexity

RUN_LIVE = os.environ.get("RUN_LIVE") == "1"
MODEL = "gpt-6.1-sol"

if RUN_LIVE:
    for name in ("OPENAI_API_KEY", "PERPLEXITY_API_KEY"):
        if not os.environ.get(name):
            raise EnvironmentError(f"Set the {name} environment variable before running this notebook.")

    openai_client = OpenAI()
    perplexity_client = Perplexity(
        default_headers={
            "X-Pplx-Integration": "openai-cookbook",
            "User-Agent": f"openai-cookbook (Perplexity/Python {perplexity.__version__})",
        }
    )
else:
    print("RUN_LIVE is not set to 1; the cells below will skip their live requests.")

## 3. Call the Search API directly

Start with one request so you can see the shape of a result. `max_results` accepts 1 to 20 and defaults to 10. Each result has `title`, `url`, and `snippet`; `date` and `last_updated` are present when the page exposes them.

In [ ]:
if RUN_LIVE:
    search = perplexity_client.search.create(
        query="Python 3.14 new features",
        max_results=5,
    )

    for result in search.results:
        print(f"{result.title}\n  {result.url}\n  date={result.date}  snippet_chars={len(result.snippet)}")

What's new in Python 3.14 — Python 3.15.2 documentation
  https://docs.python.org/3/whatsnew/3.14.html
  date=None  snippet_chars=4054
New features in Python 3.14 - Red Hat Developer
  https://developers.redhat.com/articles/2026/07/11/whats-new-in-python-3-14
  date=2026-07-11  snippet_chars=4132
What’s new in Python 3.14 — Python 3.15.0 documentation
  https://docs.python.org/3.15/whatsnew/3.14.html
  date=None  snippet_chars=3909
Python Release Python 3.14.5
  https://www.python.org/downloads/release/python-3145/
  date=2010-03-14  snippet_chars=2577
What’s new in Python 3.14 — Python 3.16.0a1 documentation
  https://docs.python.org/3.16/whatsnew/3.14.html
  date=None  snippet_chars=3734


In the output above, `date` is the publication date the index extracted for the page. It can be missing or wrong (the Python 3.14.5 release page shows `2010-03-14`), so the loop instructions in step 5 have the model rely on URLs, not dates.

## 4. Wrap the search as a function tool

The model needs two things: a Python function that runs the search and returns text, and a JSON Schema that describes when and how to call it.

The function returns a compact JSON string with the title, URL, date, and snippet of each result. `max_tokens_per_page` caps how much page text the API extracts per result, which keeps the tool output small enough to send back to the model on every turn.

In [ ]:
import json


def perplexity_web_search(query: str) -> str:
    """Search the web with the Perplexity Search API and return results as a JSON string."""
    search = perplexity_client.search.create(
        query=query,
        max_results=5,
        max_tokens_per_page=512,
    )
    results = [
        {
            "title": result.title,
            "url": result.url,
            "date": result.date,
            "snippet": result.snippet,
        }
        for result in search.results
    ]
    return json.dumps(results)


tools = [
    {
        "type": "function",
        "name": "perplexity_web_search",
        "description": (
            "Search the web for current information. Returns a JSON list of results "
            "with title, url, date, and snippet. Call it when the answer depends on "
            "recent events or facts you are not sure about."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "A specific search query, phrased like a web search.",
                }
            },
            "required": ["query"],
            "additionalProperties": False,
        },
        "strict": True,
    }
]

tool_functions = {"perplexity_web_search": perplexity_web_search}

if RUN_LIVE:
    print(perplexity_web_search("Python 3.14 new features")[:600])

[{"title": "What's new in Python 3.14 \u2014 Python 3.15.2 documentation", "url": "https://docs.python.org/3/whatsnew/3.14.html", "date": null, "snippet": "Python 3.14 is a stable release of the Python programming\nlanguage, with a mix of changes to the language, the implementation,\nand the standard library.\nThe biggest changes include template string literals, deferred evaluation of annotations, and support for subinterpreters in the standard library.\nThe library changes include significantly improved capabilities for\nintrospection in asyncio, support for Zstandard via a new `compression.


## 5. Run a tool-calling loop with the Responses API

The loop sends the question with the tool definition, runs any `function_call` items the model returns, appends each result as a `function_call_output` item, and repeats until the model answers in text. The instructions ask the model to cite returned URLs; this example does not validate citation accuracy.

The `max_turns` limit stops the loop if the model keeps searching.

In [ ]:
INSTRUCTIONS = """
Use the perplexity_web_search tool to answer questions about current events or facts you are not sure about.
Answer only from the search results. If they are insufficient, say so.
Cite sources as Markdown links whose URLs are copied exactly from the tool results.
"""


def answer_with_search(question: str, max_turns: int = 5) -> str:
    """Answer a question with the model, running Perplexity searches when the model asks for them."""
    input_list = [{"role": "user", "content": question}]

    for _ in range(max_turns):
        response = openai_client.responses.create(
            model=MODEL,
            instructions=INSTRUCTIONS,
            tools=tools,
            input=input_list,
        )
        function_calls = [item for item in response.output if item.type == "function_call"]
        if not function_calls:
            return response.output_text

        input_list += response.output
        for call in function_calls:
            arguments = json.loads(call.arguments)
            print(f"Tool call: {call.name}({arguments})")
            input_list.append(
                {
                    "type": "function_call_output",
                    "call_id": call.call_id,
                    "output": tool_functions[call.name](**arguments),
                }
            )

    raise RuntimeError(f"The model did not answer within {max_turns} turns.")


if RUN_LIVE:
    print(answer_with_search("What is the latest Python 3.14 bugfix release, and what did it change?"))

Tool call: perplexity_web_search({'query': 'site.python.org/downloads/release Python 3.14 latest bugfix release'})
Tool call: perplexity_web_search({'query': 'Python 3.14.10 October 25 2026 release changes bugfix security'})
The latest Python 3.14 bugfix release is **Python 3.14.10**, released **October 25, 2026**. It is the tenth maintenance release and an **expedited security release**, with around **120 bugfixes, build improvements, and documentation changes** from 63 contributors since 3.14.9. [Official release page](https://www.python.org/downloads/latest/python3.14/)

The headline changes are:

- **TLS security — CVE-2026-115557:** `ssl` now rejects certificate chains containing compromised eMudhra intermediate CA keys. This also protects normal `pip` use—even before `pip` itself is updated.
- **Archive extraction — CVE-2026-114583:** `tarfile.tar_filter()` now rejects link entries that resolve to the extraction destination directory itself, preventing those links from replacing 

## 6. Filter results by domain and recency

Two request parameters narrow what the model sees:

- `search_domain_filter` takes up to 20 domains. Plain domains form an allowlist; domains prefixed with `-` form a denylist. A request can use one mode or the other, not both.
- `search_recency_filter` accepts `hour`, `day`, `week`, `month`, or `year`.

This request limits results to `python.org` and its subdomains, published within the last month. To let the model choose filters, add them to the tool schema (strict mode requires every property to be listed in `required`, so declare them as nullable) and pass them through in `perplexity_web_search`.

In [ ]:
if RUN_LIVE:
    filtered = perplexity_client.search.create(
        query="Python 3.14 release",
        max_results=5,
        search_domain_filter=["python.org"],
        search_recency_filter="month",
    )

    for result in filtered.results:
        print(f"{result.date}  {result.url}")

2026-10-01  https://blog.python.org/2026/10/python-31022-31117/
2026-10-26  https://discuss.python.org/t/python-3-15-2-3-14-10-3-13-18-3-12-16-and-3-11-18-are-out/109678
2026-10-09  https://blog.python.org/authors/hugo-van-kemenade/?ref=runtimewire
2026-10-21  https://discuss.python.org/t/python-3-15-1-3-14-9-and-3-13-17-are-out/109595/1
2026-10-19  https://www.python.org/about/?pstoreid=epp?jumpid


## Next steps

- [Search API reference](https://docs.perplexity.ai/api-reference/search-post) for every request parameter, including `country`, `search_language_filter`, and date range filters.
- [Domain filter](https://docs.perplexity.ai/docs/search/filters/domain-filter) and [date and time filter](https://docs.perplexity.ai/docs/search/filters/date-time-filters) guides for path-level and date-range filtering.
- [Use Search API with the OpenAI SDK](https://docs.perplexity.ai/docs/search/agent-sdks/openai) for a tool description tuned for keyword queries and a version that accepts multiple queries per call.
- [OpenAI function calling guide](https://developers.openai.com/api/docs/guides/function-calling) for strict schemas, parallel tool calls, and streaming tool calls.